In [1]:
from Bio import SeqIO

def get_accessions(path):
    return {rec.id.split("|")[0].split("/")[0] for rec in SeqIO.parse(path, "fasta")}

acc1 = get_accessions("astro_complete_1b_genbank_mammalia_centers.faa")
acc2 = get_accessions("genbank_pcr.faa")

common = acc1 & acc2
print(f"fasta1: {len(acc1)} accessions")
print(f"fasta2: {len(acc2)} accessions")
print(f"common: {len(common)}")
print(sorted(common))

fasta1: 148 accessions
fasta2: 1223 accessions
common: 55
['MG693176', 'MN832787', 'MT734809', 'MW450837', 'MZ218053', 'OP589977', 'OP589978', 'OP589979', 'OP963600', 'OQ236134', 'OQ236136', 'OQ606244', 'OQ802698', 'OQ802706', 'OQ802712', 'OQ802747', 'OQ802749', 'OQ802750', 'OR552421', 'OR613297', 'OR613299', 'OR613303', 'OR613304', 'OR951016', 'OR951021', 'OR951022', 'OR951025', 'OR951026', 'OR951027', 'OR951032', 'OR951036', 'OR951049', 'OR951055', 'OR951068', 'OR951074', 'OR951079', 'OR951080', 'OR951082', 'OR951087', 'OR951097', 'PP711853', 'PP711855', 'PP711856', 'PP711857', 'PP711858', 'PP711872', 'PP711873', 'PP711874', 'PP711875', 'PP711876', 'PP711877', 'PP711878', 'PP711879', 'PP711880', 'PP711881']


In [6]:
from Bio import SeqIO

input_file = "genbank_mammalia_centers_pcr.faa"
output_file = "genbank_mammalia_centers_pcr_no_bats.faa"

to_remove = {
    'MG693176', 'MN832787', 'MT734809', 'MW450837', 'MZ218053', 'OP589977',
    'OP589978', 'OP589979', 'OP963600', 'OQ236134', 'OQ236136', 'OQ606244',
    'OQ802698', 'OQ802706', 'OQ802712', 'OQ802747', 'OQ802749', 'OQ802750',
    'OR552421', 'OR613297', 'OR613299', 'OR613303', 'OR613304', 'OR951016',
    'OR951021', 'OR951022', 'OR951025', 'OR951026', 'OR951027', 'OR951032',
    'OR951036', 'OR951049', 'OR951055', 'OR951068', 'OR951074', 'OR951079',
    'OR951080', 'OR951082', 'OR951087', 'OR951097', 'PP711853', 'PP711855',
    'PP711856', 'PP711857', 'PP711858', 'PP711872', 'PP711873', 'PP711874',
    'PP711875', 'PP711876', 'PP711877', 'PP711878', 'PP711879', 'PP711880',
    'PP711881',
}

kept = 0
removed = 0
with open(output_file, "w") as out:
    for rec in SeqIO.parse(input_file, "fasta"):
        # акцессия — первая часть до '|' или '/'
        acc = rec.id.split("|")[0].split("/")[0]
        if acc in to_remove:
            removed += 1
            continue
        SeqIO.write(rec, out, "fasta")
        kept += 1

print(f"removed: {removed}")
print(f"kept: {kept}")

removed: 55
kept: 93


In [7]:
from Bio import SeqIO

# Parse both files into dicts: accession -> record
# If an accession appears multiple times within one file, the first occurrence is kept.
records1 = {}
for rec in SeqIO.parse("genbank_bats_pcr_90clust.fasta", "fasta"):
    acc = rec.id.split("|")[0].split("/")[0]
    if acc not in records1:
        records1[acc] = rec

records2 = {}
for rec in SeqIO.parse("genbank_mammalia_centers_pcr_no_bats.faa", "fasta"):
    acc = rec.id.split("|")[0].split("/")[0]
    if acc not in records2:
        records2[acc] = rec

# Merge: start with all records from file1, then add records from file2
# only if the accession is NOT already present (so for common accessions,
# the version from file1 is kept).
merged = dict(records1)
for acc, rec in records2.items():
    if acc not in merged:
        merged[acc] = rec

# Write merged FASTA
with open("genbank_bats_90clust_and_mammalia_centers_pcr.faa", "w") as out:
    SeqIO.write(merged.values(), out, "fasta")

print(f"file1 unique accessions: {len(records1)}")
print(f"file2 unique accessions: {len(records2)}")
print(f"accessions in both (kept from file1): {len(set(records1) & set(records2))}")
print(f"total records written: {len(merged)}")

file1 unique accessions: 410
file2 unique accessions: 93
accessions in both (kept from file1): 0
total records written: 503


In [3]:
from Bio import SeqIO

input_file = "genbank_and_contigs_pcr_aln_filtered.fasta"

with open("contigs.faa", "w") as contigs_out, open("genbank.faa", "w") as genbank_out:
    n_contigs = 0
    n_genbank = 0
    for rec in SeqIO.parse(input_file, "fasta"):
        if "|" in rec.description:
            SeqIO.write(rec, genbank_out, "fasta")
            n_genbank += 1
        else:
            SeqIO.write(rec, contigs_out, "fasta")
            n_contigs += 1

print(f"contigs.faa: {n_contigs} sequences")
print(f"genbank.faa: {n_genbank} sequences")

contigs.faa: 98 sequences
genbank.faa: 148 sequences
